In [87]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [89]:
# 1. Güncellenmiş ana veri dosyasını oku
df = pd.read_csv("all_data_ratio_2023_2024_eklenmis.csv", encoding='utf-8-sig')

In [90]:
# 2. Tüm şehirler için tahminleri toplayacağımız liste
tahmin_edilecekler = ['uzman', 'pratisyen', 'asistan', 'disci', 'eczaci', 'ebe', 'diger', 'hemsire', 'hastane', 'yatak']

In [91]:
sehirler = df['il_adı'].unique()

In [92]:
tum_sonuclar = []

In [97]:
for sehir in sehirler:
    df_sehir = df[df['il_adı'] == sehir]

    # Eğer şehirde çok az veri varsa atla
    if df_sehir[df_sehir['yıl'] <= 2022].shape[0] < 5:
        continue

    # Bu şehir için yıllık bazlı boş veri oluştur
    sehir_yillar = pd.DataFrame({'yıl': range(2002, 2025)})
    sehir_yillar['il_adı'] = sehir

    # Nüfus verisini ekleyelim
    nufus_sehir = df_sehir[['yıl', 'nufus']]
    sehir_yillar = sehir_yillar.merge(nufus_sehir, on='yıl', how='left')

    # Şimdi uzman, pratisyen, asistan vs hepsini tahmin edeceğiz
    for hedef in tahmin_edilecekler:
        # Eğitim verisi (2002-2022)
        df_train = df_sehir[(df_sehir['yıl'] <= 2022)]

        X_train = df_train[['yıl', 'nufus']]
        y_train = df_train[hedef]

        # Eğer hedefte tamamen boş veri varsa boş bırak
        if y_train.isnull().all():
            sehir_yillar[hedef] = np.nan
            continue

        # Modeli eğit
        lr_model = LinearRegression()
        lr_model.fit(X_train.fillna(0), y_train.fillna(0))

        # 2002-2024 arası veriler
        X_full = sehir_yillar[['yıl', 'nufus']].fillna(0)
        preds = lr_model.predict(X_full)

        # Yeni sütun oluştur
        yeni_sutun = []

        for idx, yil in enumerate(sehir_yillar['yıl']):
            # Eğer 2002-2022 arası gerçek veri varsa onu kullan
            if yil <= 2022:
                deger = df_sehir[df_sehir['yıl'] == yil][hedef].values
                if len(deger) > 0 and not pd.isna(deger[0]):
                    yeni_sutun.append(deger[0])
                else:
                    yeni_sutun.append(preds[idx])
            else:
                # 2023 ve 2024 için doğrudan tahmin
                yeni_sutun.append(preds[idx])

        sehir_yillar[hedef] = yeni_sutun

    # Sonuçları ekle
    tum_sonuclar.append(sehir_yillar)

In [98]:
# 4. Tahmin sonuçlarını DataFrame yap
final_df = pd.concat(tum_sonuclar, ignore_index=True)

In [101]:
final_df.to_csv("tahminsonuclari.csv", index=False, encoding='utf-8-sig')

print("✅ Tüm şehirler için 2002-2024 tahminli veri oluşturuldu: 'sehirler_2002_2024_tahminli.csv'")

✅ Tüm şehirler için 2002-2024 tahminli veri oluşturuldu: 'sehirler_2002_2024_tahminli.csv'
